# NB01 – Data Collection

## European Air Quality and Weather

### Research question

How does air quality vary across major European cities, and which weather conditions are associated with worse pollution?

### Purpose

This notebook collects air quality and weather data for a selection of major European cities. The data is collected using the OpenWeather API.

The original API responses are saved in the `data/raw` folder so that the collection process is reproducible and the source data remains unchanged.

The collected variables include:

- Air Quality Index
- PM2.5
- PM10
- Nitrogen dioxide
- Ozone
- Carbon monoxide
- Temperature
- Humidity
- Wind speed
- Atmospheric pressure

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

## Create Project Paths

In [3]:
# The notebook is stored inside the notebooks folder,
# so ".." refers to the main final-project folder.
PROJECT_DIR = Path("..")
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

# Create the folders if they do not already exist.
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR.resolve())
print("Raw data folder:", RAW_DATA_DIR.resolve())

Project folder: /files/assignments/final-project
Raw data folder: /files/assignments/final-project/data/raw


## Load the API key

In [4]:
# Load variables stored in the .env file.
load_dotenv(PROJECT_DIR / ".env")

API_KEY = os.getenv("OPENWEATHER_API_KEY")

if API_KEY is None:
    raise ValueError(
        "The OpenWeather API key was not found. "
        "Check that the .env file exists in the final-project folder."
    )

print("API key loaded successfully.")

API key loaded successfully.


## Cities included in the project

The project compares ten major European cities from different parts of Europe. Using several cities allows the analysis to examine both variation in pollution and relationships between pollution and weather. The next thing I will be doing is defining the cities

In [5]:
cities = {
    "London": {"country": "GB", "latitude": 51.5074, "longitude": -0.1278},
    "Paris": {"country": "FR", "latitude": 48.8566, "longitude": 2.3522},
    "Berlin": {"country": "DE", "latitude": 52.5200, "longitude": 13.4050},
    "Madrid": {"country": "ES", "latitude": 40.4168, "longitude": -3.7038},
    "Rome": {"country": "IT", "latitude": 41.9028, "longitude": 12.4964},
    "Amsterdam": {"country": "NL", "latitude": 52.3676, "longitude": 4.9041},
    "Brussels": {"country": "BE", "latitude": 50.8503, "longitude": 4.3517},
    "Vienna": {"country": "AT", "latitude": 48.2082, "longitude": 16.3738},
    "Copenhagen": {"country": "DK", "latitude": 55.6761, "longitude": 12.5683},
    "Prague": {"country": "CZ", "latitude": 50.0755, "longitude": 14.4378},
}

cities_df = pd.DataFrame.from_dict(cities, orient="index")
cities_df.index.name = "city"
cities_df

,country,latitude,longitude
city,,,
London,GB,51.5074,-0.1278
Paris,FR,48.8566,2.3522
Berlin,DE,52.5200,13.4050
Madrid,ES,40.4168,-3.7038
Rome,IT,41.9028,12.4964
Amsterdam,NL,52.3676,4.9041
Brussels,BE,50.8503,4.3517
Vienna,AT,48.2082,16.3738
Copenhagen,DK,55.6761,12.5683


## Test the OpenWeather API

Before collecting data for every city, one request is tested using London. This makes it easier to identify problems with the API key, URL, or parameters.

In [6]:
WEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"

test_city = cities["London"]

weather_params = {
    "lat": test_city["latitude"],
    "lon": test_city["longitude"],
    "appid": API_KEY,
    "units": "metric"
}

weather_response = requests.get(
    WEATHER_URL,
    params=weather_params,
    timeout=30
)

print("Status code:", weather_response.status_code)

Status code: 200


## Inspect the weather response and check the individual values

In [7]:
weather_response.raise_for_status()

weather_test_data = weather_response.json()
weather_test_data

{'coord': {'lon': -0.13, 'lat': 51.51},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 27.75,
  'feels_like': 27.72,
  'temp_min': 26.67,
  'temp_max': 28.94,
  'pressure': 1014,
  'humidity': 44,
  'sea_level': 1014,
  'grnd_level': 1010},
 'visibility': 10000,
 'wind': {'speed': 2.68, 'deg': 259, 'gust': 3.13},
 'clouds': {'all': 83},
 'dt': 1785424851,
 'sys': {'type': 2,
  'id': 2075535,
  'country': 'GB',
  'sunrise': 1785385254,
  'sunset': 1785441170},
 'timezone': 3600,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

In [8]:
print("City:", weather_test_data["name"])
print("Temperature:", weather_test_data["main"]["temp"])
print("Humidity:", weather_test_data["main"]["humidity"])
print("Pressure:", weather_test_data["main"]["pressure"])
print("Wind speed:", weather_test_data["wind"]["speed"])

City: London
Temperature: 27.75
Humidity: 44
Pressure: 1014
Wind speed: 2.68


## Air pollution request

The air pollution endpoint uses the same latitude and longitude but returns the Air Quality Index and concentrations of several pollutants.

In [9]:
AIR_POLLUTION_URL = "https://api.openweathermap.org/data/2.5/air_pollution"

air_params = {
    "lat": test_city["latitude"],
    "lon": test_city["longitude"],
    "appid": API_KEY
}

air_response = requests.get(
    AIR_POLLUTION_URL,
    params=air_params,
    timeout=30
)

print("Status code:", air_response.status_code)

Status code: 200


## Inspect the response 

In [10]:
air_response.raise_for_status()

air_test_data = air_response.json()
air_test_data

{'coord': {'lon': -0.1276, 'lat': 51.5072},
 'list': [{'main': {'aqi': 2},
   'components': {'co': 96.92,
    'no': 1.16,
    'no2': 4.53,
    'o3': 88.38,
    'so2': 5.31,
    'pm2_5': 6.35,
    'pm10': 14.93,
    'nh3': 0},
   'dt': 1785424846}]}

## Examine the air quality values

In [11]:
air_record = air_test_data["list"][0]

print("AQI category:", air_record["main"]["aqi"])
print("PM2.5:", air_record["components"]["pm2_5"])
print("PM10:", air_record["components"]["pm10"])
print("Nitrogen dioxide:", air_record["components"]["no2"])
print("Ozone:", air_record["components"]["o3"])
print("Carbon monoxide:", air_record["components"]["co"])

AQI category: 2
PM2.5: 6.35
PM10: 14.93
Nitrogen dioxide: 4.53
Ozone: 88.38
Carbon monoxide: 96.92


## Collect data for one city

The function below sends one weather request and one air pollution request for a city. It checks that both requests are successful and returns the original JSON responses.

In [13]:
def collect_city_data(city_name, city_info):
    """
    Collect current weather and air-quality data for one city.

    Parameters
    ----------
    city_name : str
        Name of the city.
    city_info : dict
        Dictionary containing country, latitude, and longitude.

    Returns
    -------
    dict
        Original weather and air-pollution API responses together
        with collection metadata.
    """
    
    common_params = {
        "lat": city_info["latitude"],
        "lon": city_info["longitude"],
        "appid": API_KEY
    }
    
    weather_params = common_params.copy()
    weather_params["units"] = "metric"
    
    weather_response = requests.get(
        WEATHER_URL,
        params=weather_params,
        timeout=30
    )
    
    air_response = requests.get(
        AIR_POLLUTION_URL,
        params=common_params,
        timeout=30
    )
    
    weather_response.raise_for_status()
    air_response.raise_for_status()
    
    return {
        "city_requested": city_name,
        "country": city_info["country"],
        "latitude": city_info["latitude"],
        "longitude": city_info["longitude"],
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
        "weather": weather_response.json(),
        "air_pollution": air_response.json()
    }

## Test the function using London

In [14]:
london_test = collect_city_data("London", cities["London"])

print(london_test.keys())
print(london_test["city_requested"])
print(london_test["collected_at_utc"])

dict_keys(['city_requested', 'country', 'latitude', 'longitude', 'collected_at_utc', 'weather', 'air_pollution'])
London
2026-07-30T15:29:46.971843+00:00


## Collect data for all selected cities

The function is now applied to every city. Each result contains collection metadata, the full weather response, and the full air pollution response.

In [15]:
all_city_data = []

for city_name, city_info in cities.items():
    try:
        city_data = collect_city_data(city_name, city_info)
        all_city_data.append(city_data)
        print(f"Collected data for {city_name}")
        
    except requests.RequestException as error:
        print(f"Could not collect data for {city_name}: {error}")

print(f"\nSuccessfully collected {len(all_city_data)} cities.")

Collected data for London
Collected data for Paris
Collected data for Berlin
Collected data for Madrid
Collected data for Rome
Collected data for Amsterdam
Collected data for Brussels
Collected data for Vienna
Collected data for Copenhagen
Collected data for Prague

Successfully collected 10 cities.


## Save the raw API responses

The complete API responses are saved before any cleaning or transformation. Keeping the raw data unchanged makes the project reproducible and allows the processing steps to be checked later.

In [16]:
collection_time = datetime.now(timezone.utc)
file_timestamp = collection_time.strftime("%Y%m%d_%H%M%S")

raw_output_path = (
    RAW_DATA_DIR /
    f"european_air_quality_weather_{file_timestamp}.json"
)

with open(raw_output_path, "w", encoding="utf-8") as file:
    json.dump(all_city_data, file, indent=4)

print("Raw data saved to:")
print(raw_output_path.resolve())

Raw data saved to:
/files/assignments/final-project/data/raw/european_air_quality_weather_20260730_153028.json


## Check that the saved JSON can be reopened

In [17]:
with open(raw_output_path, "r", encoding="utf-8") as file:
    saved_data_check = json.load(file)

print("Number of saved city records:", len(saved_data_check))
print("First saved city:", saved_data_check[0]["city_requested"])

Number of saved city records: 10
First saved city: London


## Create a collection summary 
The detailed cleaning will be done in NB02 however this step helps to confirm that all the correct variables have been collected

In [18]:
collection_summary = []

for record in all_city_data:
    weather = record["weather"]
    air = record["air_pollution"]["list"][0]
    components = air["components"]
    
    collection_summary.append({
        "city": record["city_requested"],
        "country": record["country"],
        "collected_at_utc": record["collected_at_utc"],
        "temperature_c": weather["main"]["temp"],
        "humidity_percent": weather["main"]["humidity"],
        "pressure_hpa": weather["main"]["pressure"],
        "wind_speed_ms": weather["wind"]["speed"],
        "aqi": air["main"]["aqi"],
        "pm2_5": components["pm2_5"],
        "pm10": components["pm10"],
        "no2": components["no2"],
        "o3": components["o3"],
        "co": components["co"]
    })

collection_summary_df = pd.DataFrame(collection_summary)
collection_summary_df

,city,country,collected_at_utc,temperature_c,humidity_percent,pressure_hpa,wind_speed_ms,aqi,pm2_5,pm10,no2,o3,co
0,London,GB,2026-07-30T15:30:07.388303+00:00,27.75,44,1014,2.68,2,6.35,14.93,4.53,88.38,96.92
1,Paris,FR,2026-07-30T15:30:07.466908+00:00,30.24,51,1016,3.75,3,8.85,9.88,0.63,103.97,115.92
2,Berlin,DE,2026-07-30T15:30:07.552969+00:00,38.02,22,1009,2.68,3,4.79,5.53,3.00,123.64,101.60
3,Madrid,ES,2026-07-30T15:30:07.636580+00:00,37.33,17,1015,0.45,3,18.59,76.88,0.17,87.23,91.13
4,Rome,IT,2026-07-30T15:30:07.729556+00:00,35.98,33,1015,2.68,3,11.66,22.97,0.35,117.59,170.53
5,Amsterdam,NL,2026-07-30T15:30:07.809257+00:00,24.50,64,1015,0.89,3,10.96,14.68,7.01,102.66,124.39
6,Brussels,BE,2026-07-30T15:30:07.887855+00:00,22.62,81,1016,4.33,2,9.24,11.28,1.22,93.19,107.92
7,Vienna,AT,2026-07-30T15:30:07.955422+00:00,31.71,30,1017,3.36,3,7.06,7.74,1.43,116.18,103.76
8,Copenhagen,DK,2026-07-30T15:30:08.035749+00:00,28.97,50,1011,5.85,3,10.43,11.48,6.38,123.94,133.38
9,Prague,CZ,2026-07-30T15:30:08.115697+00:00,36.54,21,1015,3.13,3,3.72,4.37,1.66,104.94,91.82


## Run basic collection checks

In [19]:
print("Shape:", collection_summary_df.shape)
print("\nColumns:")
print(collection_summary_df.columns.tolist())

print("\nMissing values:")
print(collection_summary_df.isna().sum())

Shape: (10, 13)

Columns:
['city', 'country', 'collected_at_utc', 'temperature_c', 'humidity_percent', 'pressure_hpa', 'wind_speed_ms', 'aqi', 'pm2_5', 'pm10', 'no2', 'o3', 'co']

Missing values:
city                0
country             0
collected_at_utc    0
temperature_c       0
humidity_percent    0
pressure_hpa        0
wind_speed_ms       0
aqi                 0
pm2_5               0
pm10                0
no2                 0
o3                  0
co                  0
dtype: int64


## Save the collection log

In [20]:
collection_log_path = (
    RAW_DATA_DIR /
    f"collection_log_{file_timestamp}.csv"
)

collection_summary_df.to_csv(collection_log_path, index=False)

print("Collection log saved to:")
print(collection_log_path.resolve())

Collection log saved to:
/files/assignments/final-project/data/raw/collection_log_20260730_153028.csv


## Data collection summary

This notebook successfully collected current weather and air quality data for ten major European cities using two OpenWeather API endpoints.

For each city, the collection includes:

- temperature
- humidity
- atmospheric pressure
- wind speed
- Air Quality Index
- PM2.5
- PM10
- nitrogen dioxide
- ozone
- carbon monoxide

The complete original API responses were saved as JSON in `data/raw`. A CSV collection log was also saved to make it easier to verify the collected observations.

The next notebook will load the raw JSON data, flatten its nested structure, check data quality, add understandable AQI labels, and save a tidy processed dataset.